# Notebook 01 — Dataset Acquisition and Provenance

**AI Interview Assistant · Machine Learning Pipeline, Stage 1 of 9**

---

## Purpose

Before any modelling decision can be defended, the data behind it has to be
traceable. This notebook acquires the interview-question corpus, records exactly
where every record came from, and proves that no pretrained model weights
entered the project.

## Why this stage exists

A dissertation-grade pipeline must answer three questions about its data:

| Question | How this notebook answers it |
|---|---|
| *Where did the data come from?* | Source identifier, URL and licence recorded per record |
| *Has it changed since?* | SHA-256 hash of the raw file, written to a provenance report |
| *Is the model genuinely ours?* | An audit that fails loudly if any pretrained weight file is present |

## Project constraint — no pretrained models

The system is required to use **only models trained from scratch within this
project**. Hugging Face is therefore used strictly as a *dataset* host. Stage 6
of this notebook enforces that: it scans the workspace for weight files
(`.safetensors`, `.bin`, `.onnx`, `.h5`) and raises if any are found.

## Inputs and outputs

- **Input** — `ali-alkhars/interviews` (Hugging Face dataset repository)
- **Output** — `dataset/raw/raw_interview_dataset.json`
- **Output** — `reports/dataset_metadata.json` (provenance record)
- **Output** — `reports/figures/01_*.png` (acquisition summary figures)

---

In [ ]:
NOTEBOOK_ID = 1

# ─────────────────────────────────────────────────────────────────────────────
# Step 0 — Environment bootstrap
#
# Locates the project workspace so this notebook runs unchanged in Google Colab,
# a local Jupyter server, or VS Code. Every later step resolves its paths from
# WORKSPACE_DIR, so nothing below depends on where the notebook was opened.
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def locate_workspace() -> Path:
    """Return the ml-service directory, whatever environment we are in."""
    # 1. Google Colab: mount Drive so checkpoints survive a runtime restart.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        ws = Path("/content/drive/MyDrive/ai-interview-system/ml-service")
        ws.mkdir(parents=True, exist_ok=True)
        print("Environment      : Google Colab (Drive mounted)")
        return ws
    except ImportError:
        pass

    # 2. Local: walk up from the notebook until we find the ml-service root,
    #    identified by the dataset directory it must contain.
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            print("Environment      : local")
            return candidate
    print("Environment      : local (fallback to cwd)")
    return here

WORKSPACE_DIR = locate_workspace()
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# Canonical paths used across all nine notebooks.
RAW_DIR       = WORKSPACE_DIR / "dataset" / "raw"
PROCESSED_DIR = WORKSPACE_DIR / "dataset" / "processed"
QG_DIR        = PROCESSED_DIR / "question_generator"
SPLIT_DIR     = PROCESSED_DIR / "splits"
TOKENIZER_DIR = WORKSPACE_DIR / "tokenizer"
CKPT_DIR      = WORKSPACE_DIR / "checkpoints"
MODEL_DIR     = WORKSPACE_DIR / "models"
REPORTS_DIR   = WORKSPACE_DIR / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"

for d in (RAW_DIR, PROCESSED_DIR, SPLIT_DIR, TOKENIZER_DIR, CKPT_DIR,
          MODEL_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace        : {WORKSPACE_DIR}")
print(f"Python           : {sys.version.split()[0]}")
print(f"Run started      : {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

---

## Step 1 — Hardware preflight

Training in Stage 5 is only feasible on some of the environments this notebook
runs in. Recording the hardware here means every later result can be attributed
to the machine that produced it, and the training notebook can size its batches
accordingly rather than crashing on a small GPU.

In [ ]:
import platform

hardware = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "processor": platform.processor() or "unknown",
}

try:
    import torch
    hardware["torch_version"] = torch.__version__
    hardware["cuda_available"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        hardware["gpu_name"] = torch.cuda.get_device_name(0)
        hardware["gpu_memory_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
        hardware["gpu_count"] = torch.cuda.device_count()
    else:
        hardware["gpu_name"] = None
        hardware["gpu_memory_gb"] = 0.0
        hardware["gpu_count"] = 0
except ImportError:
    hardware["torch_version"] = None
    hardware["cuda_available"] = False

try:
    import psutil
    hardware["cpu_cores_physical"] = psutil.cpu_count(logical=False)
    hardware["cpu_cores_logical"] = psutil.cpu_count(logical=True)
    hardware["ram_total_gb"] = round(psutil.virtual_memory().total / 1024**3, 2)
except ImportError:
    hardware["cpu_cores_physical"] = os.cpu_count()
    hardware["ram_total_gb"] = None

# The training device chosen here is what Stage 5 will actually use.
DEVICE = "cuda" if hardware.get("cuda_available") else "cpu"
hardware["selected_device"] = DEVICE

print("HARDWARE PREFLIGHT")
print("=" * 62)
for key, value in hardware.items():
    print(f"  {key:24s} : {value}")
print("=" * 62)
print(f"\nTraining device for Stage 5: {DEVICE}")
if DEVICE == "cpu":
    print("Note: no GPU detected. Stage 5 will train the compact architectures\n"
          "      only; the scaled variants will be skipped as infeasible.")

---

## Step 2 — Download the raw dataset

The corpus is `ali-alkhars/interviews`, a public collection of technical
interview questions labelled by domain and difficulty. It is fetched from the
dataset endpoint — not the model endpoint — which is what keeps the
no-pretrained-weights guarantee intact.

The download is **idempotent**: if the file already exists locally it is reused,
so re-running the notebook does not depend on network access.

In [ ]:
import urllib.request
import urllib.error

HF_DATASET_ID = "ali-alkhars/interviews"
HF_DATASET_URL = (
    "https://huggingface.co/datasets/ali-alkhars/interviews/raw/main/interviews.json"
)
RAW_FILE = RAW_DIR / "raw_interview_dataset.json"

def load_raw_records() -> list:
    """Return the raw records, downloading them only if not already present."""
    if RAW_FILE.exists() and RAW_FILE.stat().st_size > 1024:
        print(f"Reusing cached download: {RAW_FILE.name} "
              f"({RAW_FILE.stat().st_size / 1024:.1f} KB)")
        return json.loads(RAW_FILE.read_text(encoding="utf-8"))

    print(f"Downloading {HF_DATASET_ID} ...")
    request = urllib.request.Request(
        HF_DATASET_URL, headers={"User-Agent": "ai-interview-system/1.0"})
    with urllib.request.urlopen(request, timeout=90) as response:
        payload = json.loads(response.read().decode("utf-8"))

    records = payload if isinstance(payload, list) else payload.get("data", [])
    RAW_FILE.write_text(json.dumps(records, indent=1, ensure_ascii=False),
                        encoding="utf-8")
    print(f"Downloaded {len(records)} records -> {RAW_FILE.name}")
    return records

records = load_raw_records()

print(f"\nRecords loaded    : {len(records):,}")
print(f"Raw file size     : {RAW_FILE.stat().st_size / 1024:.1f} KB")
print(f"\nFirst record:")
print(json.dumps(records[0], indent=2, ensure_ascii=False)[:600])

---

## Step 3 — Schema and integrity assertions

Rather than trusting the download, each structural assumption the rest of the
pipeline depends on is asserted here. A failure at this point is far cheaper to
diagnose than a silent `KeyError` inside a training loop three notebooks later.

In [ ]:
assert isinstance(records, list), "Expected a JSON array of records"
assert len(records) > 0, "Dataset is empty — the download failed"

# Which fields actually appear, and how often.
field_counts = {}
for record in records:
    for field in record:
        field_counts[field] = field_counts.get(field, 0) + 1

schema = pd.DataFrame({
    "field": list(field_counts),
    "present_in": list(field_counts.values()),
})
schema["coverage_%"] = (schema["present_in"] / len(records) * 100).round(2)
schema["example"] = [
    str(next((r[f] for r in records if r.get(f) not in (None, "")), ""))[:52]
    for f in schema["field"]
]
schema = schema.sort_values("coverage_%", ascending=False).reset_index(drop=True)

print("DATASET SCHEMA")
print("=" * 78)
print(schema.to_string(index=False))
print("=" * 78)

# The pipeline cannot run without a non-empty question on every record.
missing_question = [i for i, r in enumerate(records)
                    if not str(r.get("question", "")).strip()]
assert not missing_question, (
    f"{len(missing_question)} records have no question text "
    f"(first at index {missing_question[0]})"
)
print(f"\nAssertion passed: all {len(records):,} records carry question text.")

for required in ("domain", "difficulty"):
    coverage = field_counts.get(required, 0) / len(records) * 100
    print(f"Label coverage — {required:11s}: {coverage:6.2f}%")

In [ ]:
NOTEBOOK_ID = 1

# ─────────────────────────────────────────────────────────────────────────────
# Step 0b — Plotting conventions
#
# One style definition for every figure in the pipeline, so figures across the
# nine notebooks are directly comparable in the dissertation. Every figure is
# also saved to reports/figures/ at 200 dpi, ready to drop into the write-up.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#444444",
    "grid.alpha": 0.3,
    "legend.frameon": True,
    "figure.autolayout": False,
})

# Colour-blind-safe categorical palette, used consistently for every chart.
PALETTE = ["#3B6FD4", "#E1893B", "#3EA37A", "#C4576B", "#7B5EA7", "#8C7B68"]
sns.set_palette(PALETTE)

_figure_index = {"n": 0}

def save_figure(fig, slug: str, caption: str = "") -> Path:
    """Save a figure with a numbered filename and print its caption."""
    _figure_index["n"] += 1
    n = _figure_index["n"]
    path = FIGURES_DIR / f"{NOTEBOOK_ID:02d}_fig{n:02d}_{slug}.png"
    fig.savefig(path)
    label = f"Figure {NOTEBOOK_ID}.{n}"
    if caption:
        print(f"{label}: {caption}")
    print(f"           saved -> {path.relative_to(WORKSPACE_DIR)}")
    return path

def describe_series(series: pd.Series, name: str) -> pd.Series:
    """Summary statistics reported in a consistent order for every variable."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "min": s.min(),
        "q1": s.quantile(0.25),
        "median": s.median(),
        "q3": s.quantile(0.75),
        "max": s.max(),
        "skew": s.skew(),
        "kurtosis": s.kurtosis(),
    }, name=name)

print("Plot style       : configured")
print(f"Figure output    : {FIGURES_DIR.relative_to(WORKSPACE_DIR)}")
print(f"Palette          : {len(PALETTE)} colour-blind-safe categories")

---

## Step 4 — Visualise what was acquired

Three figures characterise the raw corpus before any cleaning. Their purpose is
diagnostic: **class imbalance visible here dictates the sampling strategy in
Stage 3 and the stratification in Stage 4.**

In [ ]:
raw_df = pd.DataFrame(records)
raw_df["question"] = raw_df["question"].astype(str)
raw_df["q_char_len"] = raw_df["question"].str.len()
raw_df["q_word_count"] = raw_df["question"].str.split().str.len()

# ── Figure 1.1 — question length distribution ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(raw_df["q_char_len"], bins=50, color=PALETTE[0],
             edgecolor="white", linewidth=0.5)
axes[0].axvline(raw_df["q_char_len"].mean(), color=PALETTE[3], linestyle="--",
                linewidth=2, label=f"mean = {raw_df['q_char_len'].mean():.0f}")
axes[0].axvline(raw_df["q_char_len"].median(), color=PALETTE[2], linestyle=":",
                linewidth=2, label=f"median = {raw_df['q_char_len'].median():.0f}")
axes[0].set_title("Question length (characters)")
axes[0].set_xlabel("Characters per question")
axes[0].set_ylabel("Number of questions")
axes[0].legend()

axes[1].hist(raw_df["q_word_count"], bins=40, color=PALETTE[1],
             edgecolor="white", linewidth=0.5)
axes[1].axvline(raw_df["q_word_count"].mean(), color=PALETTE[3], linestyle="--",
                linewidth=2, label=f"mean = {raw_df['q_word_count'].mean():.1f}")
axes[1].set_title("Question length (words)")
axes[1].set_xlabel("Words per question")
axes[1].set_ylabel("Number of questions")
axes[1].legend()

fig.suptitle("Raw corpus — question length distributions", y=1.02,
             fontsize=14, fontweight="bold")
save_figure(fig, "raw_length_hist",
            "Question length in the raw corpus. Both distributions are "
            "right-skewed, so length caps in Stage 3 are set by percentile "
            "rather than by mean +/- sd.")
plt.show()

print(describe_series(raw_df["q_word_count"], "q_word_count").round(2).to_string())

In [ ]:
# ── Figure 1.2 — label balance ───────────────────────────────────────────────
label_cols = [c for c in ("domain", "difficulty") if c in raw_df.columns]
fig, axes = plt.subplots(1, len(label_cols), figsize=(7 * len(label_cols), 4.6))
if len(label_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, label_cols):
    counts = raw_df[col].fillna("(missing)").value_counts()
    top = counts.head(12)
    bars = ax.barh(top.index[::-1], top.values[::-1], color=PALETTE[0])
    ax.set_title(f"{col.title()} distribution  (n = {len(raw_df):,})")
    ax.set_xlabel("Number of questions")
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.margins(x=0.12)

    # Imbalance ratio is the number the sampling strategy has to answer for.
    ratio = counts.max() / max(counts.min(), 1)
    ax.annotate(f"imbalance ratio = {ratio:.1f}:1",
                xy=(0.98, 0.03), xycoords="axes fraction", ha="right",
                fontsize=9, style="italic",
                bbox=dict(boxstyle="round,pad=0.35", fc="#FFF4E5", ec="#E1893B"))

fig.suptitle("Raw corpus — class balance across label columns", y=1.03,
             fontsize=14, fontweight="bold")
save_figure(fig, "raw_label_balance",
            "Class balance per label column. The imbalance ratios shown are "
            "what Stage 4 stratifies against.")
plt.show()

In [ ]:
# ── Figure 1.3 — source composition ──────────────────────────────────────────
source_col = "source" if "source" in raw_df.columns else None
if source_col:
    counts = raw_df[source_col].fillna("(unrecorded)").value_counts()
    fig, ax = plt.subplots(figsize=(7, 5))
    wedges, _, autotexts = ax.pie(
        counts.values, labels=None, autopct="%1.1f%%", startangle=110,
        colors=PALETTE[: len(counts)],
        wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2),
        pctdistance=0.78,
    )
    for t in autotexts:
        t.set_fontsize(9)
        t.set_fontweight("bold")
    ax.legend(wedges, [f"{s}  (n={c:,})" for s, c in counts.items()],
              loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9)
    ax.set_title("Provenance — records per source", pad=16)
    save_figure(fig, "raw_source_mix",
                "Every record is attributed to a named, licensed source.")
    plt.show()
else:
    print("No `source` column present; provenance is recorded at file level "
          "in Step 6 instead.")

print(f"\nDistinct domains     : {raw_df['domain'].nunique() if 'domain' in raw_df else 0}")
print(f"Distinct difficulties: {raw_df['difficulty'].nunique() if 'difficulty' in raw_df else 0}")
print(f"Exact duplicate questions: "
      f"{raw_df['question'].str.strip().str.lower().duplicated().sum():,}")

---

## Step 5 — Answer-field quality check

The corpus ships an `answer` field. Before Stage 5 trains on it, we check
whether it carries real content — because if it does not, training on
question + answer teaches the model a template rather than knowledge.

This check exists because it found a genuine problem: the answers are
formulaic, so the pipeline treats this as a **question-generation** corpus and
does not train on the answer text as ground truth.

In [ ]:
if "answer" in raw_df.columns:
    answers = raw_df["answer"].fillna("").astype(str)

    # A templated answer restates the question inside a fixed wrapper.
    templated = answers.str.contains(
        r"^(?:detailed|comprehensive|brief)\s+(?:technical\s+)?"
        r"(?:explanation|answer|description)\s+of\b",
        case=False, regex=True, na=False)

    distinct_ratio = answers.str.strip().str.lower().nunique() / max(len(answers), 1)
    mean_len = answers.str.split().str.len().mean()

    print("ANSWER FIELD QUALITY")
    print("=" * 62)
    print(f"  Non-empty answers        : {(answers.str.strip() != '').sum():,}"
          f" / {len(answers):,}")
    print(f"  Template-phrased answers : {templated.sum():,}"
          f"  ({templated.mean() * 100:.1f}%)")
    print(f"  Distinct-answer ratio    : {distinct_ratio:.3f}")
    print(f"  Mean answer length       : {mean_len:.1f} words")
    print("=" * 62)
    print("\nExample answers:")
    for text in answers[answers.str.strip() != ""].head(3):
        print(f"  - {text[:96]}")

    USE_ANSWERS_AS_TARGETS = bool(templated.mean() < 0.25 and mean_len > 20)
    print(f"\nDecision: train on the answer field as ground truth? "
          f"{'YES' if USE_ANSWERS_AS_TARGETS else 'NO'}")
    if not USE_ANSWERS_AS_TARGETS:
        print("Reason  : the answers are template-generated restatements of the\n"
              "          question, not real explanations. Using them as targets\n"
              "          would train the model to emit the template. Stages 3-5\n"
              "          therefore model question *generation* only, and the\n"
              "          runtime scores answers with the metrics in Stage 8.")
else:
    USE_ANSWERS_AS_TARGETS = False
    print("No `answer` column present — question generation only.")

---

## Step 6 — Zero-pretrained-weights audit

The project constraint is only credible if it is *tested*. This step walks the
whole workspace looking for model weight files. Any hit is a hard failure.

`.pt` checkpoints produced by our own Stage 5 training are explicitly allowed —
they are outputs of this project, not imports into it.

In [ ]:
FORBIDDEN_EXTENSIONS = {".safetensors", ".bin", ".onnx", ".h5", ".pb", ".tflite"}
# Directories that legitimately contain third-party binaries unrelated to models.
SKIP_DIRS = {"venv", ".venv", "node_modules", "__pycache__", ".git",
             "site-packages", ".ipynb_checkpoints"}

violations = []
own_checkpoints = []

for root, dirnames, filenames in os.walk(WORKSPACE_DIR):
    dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
    for filename in filenames:
        suffix = Path(filename).suffix.lower()
        full = Path(root) / filename
        if suffix in FORBIDDEN_EXTENSIONS:
            violations.append(full.relative_to(WORKSPACE_DIR))
        elif suffix == ".pt":
            own_checkpoints.append(full.relative_to(WORKSPACE_DIR))

print("ZERO-PRETRAINED-WEIGHTS AUDIT")
print("=" * 70)
print(f"  Scanned root            : {WORKSPACE_DIR}")
print(f"  Forbidden extensions    : {', '.join(sorted(FORBIDDEN_EXTENSIONS))}")
print(f"  Violations found        : {len(violations)}")
print(f"  Own .pt checkpoints     : {len(own_checkpoints)} (permitted — our output)")
print("=" * 70)

if violations:
    for path in violations[:20]:
        print(f"  VIOLATION: {path}")

assert not violations, (
    f"Zero-pretrained-model policy violated: {len(violations)} weight file(s) "
    f"found. First: {violations[0] if violations else ''}"
)
print("\nAUDIT PASSED — the corpus is data only; no pretrained weights present.")

---

## Step 7 — Write the provenance record

The provenance file is the artefact a reader of the dissertation would use to
reproduce this stage. It pins the dataset identity, the exact bytes downloaded
(SHA-256), the hardware, and the audit result.

In [ ]:
import hashlib

raw_bytes = RAW_FILE.read_bytes()
provenance = {
    "stage": "01_dataset_acquisition",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "dataset": {
        "identifier": HF_DATASET_ID,
        "url": HF_DATASET_URL,
        "access_type": "dataset repository (no model weights)",
        "licence": "see dataset card",
        "record_count": len(records),
        "file": RAW_FILE.name,
        "file_bytes": len(raw_bytes),
        "sha256": hashlib.sha256(raw_bytes).hexdigest(),
    },
    "label_columns": {
        col: int(raw_df[col].nunique())
        for col in ("domain", "difficulty") if col in raw_df.columns
    },
    "question_length_words": describe_series(
        raw_df["q_word_count"], "q_word_count").round(3).to_dict(),
    "answers_used_as_targets": bool(USE_ANSWERS_AS_TARGETS),
    "hardware": hardware,
    "policy_audit": {
        "forbidden_extensions": sorted(FORBIDDEN_EXTENSIONS),
        "violations": [str(p) for p in violations],
        "passed": not violations,
    },
}

metadata_path = REPORTS_DIR / "dataset_metadata.json"
metadata_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")

print("PROVENANCE RECORD WRITTEN")
print("=" * 70)
print(f"  Path      : {metadata_path.relative_to(WORKSPACE_DIR)}")
print(f"  SHA-256   : {provenance['dataset']['sha256']}")
print(f"  Records   : {provenance['dataset']['record_count']:,}")
print("=" * 70)

---

## Stage 1 summary

| Check | Result |
|---|---|
| Dataset downloaded and cached | `dataset/raw/raw_interview_dataset.json` |
| Schema assertions | every record carries question text |
| Class balance measured | imbalance ratios recorded per label column |
| Answer field audited | templated — **not** used as training targets |
| Zero-pretrained-weights audit | passed |
| Provenance recorded | `reports/dataset_metadata.json` with SHA-256 |

### What the figures told us, and what changes because of it

1. **Question lengths are right-skewed** (Figure 1.1) → Stage 3 filters by
   percentile, not by mean ± standard deviation.
2. **The label classes are imbalanced** (Figure 1.2) → Stage 4 uses
   *stratified* splitting so rare domains appear in every split.
3. **The answer field is template-generated** (Step 5) → the pipeline models
   question generation, and answer quality is scored at runtime by the metrics
   built in Stage 8 rather than learned from this corpus.

### Next

**Notebook 02 — Exploratory Data Analysis**, which characterises the corpus in
depth before any transformation is applied.